# GA/SA 特徵篩選專案使用說明

這個資料夾是「非 all-in-one」版本的完整專案包。它保留 Python 檔案分工，不把所有程式塞進一個 notebook，因此比較容易維護，也比較不容易遇到 notebook 原始 JSON 編碼問題。

## 這包裡有什麼

```text
GA_SA_feature_selection_project/
  GA_SA_feature_selection.ipynb
  run_experiment.py
  run_multi_seed.py
  aggregate_results.py
  run_smoke.ps1
  run_quick_real.ps1
  run_multi_seed_quick.ps1
  requirements.txt
  README_zh_TW.md
  PYTHON流程說明.md
  src/ml_feature_selection/
  data/raw/
  data/cache/
  results/run_20260611_153207/
```

重點檔案：

- `GA_SA_feature_selection.ipynb`：Notebook 入口，會呼叫這包裡的 `.py` 檔。
- `run_experiment.py`：單次實驗主程式。
- `run_multi_seed.py`：多 seed 批次實驗。
- `aggregate_results.py`：彙總多個 `results/run_*`。
- `PYTHON流程說明.md`：更細的程式流程解釋。
- `results/run_20260611_153207/`：已跑完的 standard 結果範例。

## 先放資料

因為 Kaggle CSV 很大，這個 package 沒有複製原始資料。請把：

```text
train_transaction.csv
```

放到：

```text
GA_SA_feature_selection_project/data/raw/train_transaction.csv
```

目前程式沒有合併 `train_identity.csv`，只使用 `train_transaction.csv`。

## 建議執行順序

先進入 package 資料夾：

```powershell
cd D:\ML期末\GA_SA_feature_selection_project
```

### 1. 先跑 smoke 測試

這不需要 Kaggle 資料，用 synthetic data 確認整套流程能跑。

```powershell
powershell -ExecutionPolicy Bypass -File .\run_smoke.ps1
```

如果成功，會產生：

```text
results/run_時間/
```

### 2. 跑正式 quick

確認 `data/raw/train_transaction.csv` 已經放好後執行：

```powershell
powershell -ExecutionPolicy Bypass -File .\run_quick_real.ps1
```

`quick` 是筆電優先設定：

```text
proxy_rows = 20000
search_features = 80
lambda = 0.05, 0.10
GA generations = 18
SA iterations = 600
final CV = 3-fold
```

### 3. 跑多 seed

如果要讓報告更有說服力，跑：

```powershell
powershell -ExecutionPolicy Bypass -File .\run_multi_seed_quick.ps1
```

它會跑：

```text
seed = 42, 7, 13
preset = quick
```

跑完會多一個：

```text
results/aggregate_時間/
```

裡面有平均與標準差。

## 用 Notebook 跑

打開：

```text
GA_SA_feature_selection.ipynb
```

Notebook 預設只跑 smoke，不會自動跑正式資料。

要跑正式資料時，找到正式資料那格，改成：

```python
RUN_REAL = True
```

要跑多 seed 時，找到多 seed 那格，改成：

```python
RUN_MULTI_SEED = True
```

如果只是想看目前結果，直接跑「查看最新結果」和「查看多 Seed 彙總結果」的 cell 即可。

## 輸出結果怎麼看

每次實驗會輸出到：

```text
results/run_YYYYMMDD_HHMMSS/
```

主要檔案：

```text
summary.md
metrics.csv
selected_features.json
convergence.csv
plots/convergence.png
plots/pareto_auc_vs_features.png
plots/lambda_sensitivity.png
run_metadata.json
```

最重要的是 `metrics.csv` 裡的：

```text
final_cv_auc
final_cv_fitness
selected_count
```

不要只看 inner-loop 的 `auc`，因為 GA/SA 在搜尋階段可能對 holdout validation 有一點 overfitting。

## 已附的 standard 結果

這包裡附了一份已跑完的結果：

```text
results/run_20260611_153207/
```

它的設定是：

```text
preset = standard
rows = 20000
search_features = 120
lambda = 0.0, 0.05, 0.10, 0.20
final CV = 5-fold
```

重點結論：

```text
lambda = 0.05：SA final AUC 約 0.8049，54 個特徵
lambda = 0.10：GA final AUC 約 0.8045，43 個特徵
AllSearchFeatures final AUC 約 0.8024，120 個特徵
```

可以寫成：

```text
GA/SA 能用較少特徵達到接近或略高於全部搜尋特徵的 final CV AUC。
```

不要寫成：

```text
GA/SA 絕對大幅勝過全部特徵。
```

因為 AUC 差距不大。

## Cache 說明

正式資料前處理會自動 cache 到：

```text
data/cache/
```

第一次跑會比較久，因為要讀大 CSV、補缺失值、類別編碼、Variance Threshold、MI 預篩。

之後如果資料與參數相同，就會直接讀 cache。

如果要強制重建：

```powershell
python run_experiment.py --mode real --data-path .\data\raw\train_transaction.csv --preset quick --no-cache
```

## 編碼注意

這個資料夾的中文說明檔是 UTF-8。建議用 VS Code、Jupyter、Notepad、Typora 或 Obsidian 開啟。

如果 PowerShell 顯示中文路徑亂碼，請用這些 `.ps1` 腳本執行，因為它們已經設定：

```text
PYTHONUTF8=1
PYTHONIOENCODING=utf-8
chcp 65001
```

## 建議交作業流程

1. 先用 `results/run_20260611_153207/` 寫初版結果。
2. 如果時間夠，跑 `run_multi_seed_quick.ps1`。
3. 用 `aggregate_summary.md` 補穩定性分析。
4. 報告中引用 final CV，不引用 inner AUC 當最終結論。



## 一、環境設定


In [2]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
print('專案根目錄：', PROJECT_ROOT)

os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

required_files = [
    Path('run_experiment.py'),
    Path('run_multi_seed.py'),
    Path('aggregate_results.py'),
    Path('src/ml_feature_selection'),
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('請從專案資料夾開啟這份 notebook。缺少：' + ' ' + ', '.join(missing_files))

def run_stream(cmd):
    env = os.environ.copy()
    env['PYTHONUTF8'] = '1'
    env['PYTHONIOENCODING'] = 'utf-8'
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)



專案根目錄： d:\ML期末\GA_SA_feature_selection_project


## 二、檢查套件


In [3]:
import importlib.util
import subprocess
import sys

packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'xgboost': 'xgboost',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
}

missing_packages = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print('正在安裝缺少套件：', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('必要套件都已安裝。')


必要套件都已安裝。


## 三、跑 smoke 測試


In [ ]:
import subprocess
import sys

cmd = [sys.executable, '-u', 'run_experiment.py', '--mode', 'synthetic', '--preset', 'smoke', '--device', 'cpu']
print('執行指令：', ' '.join(cmd))
run_stream(cmd)


## 四、查看最新單次結果


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

runs = sorted(Path('results').glob('run_*'))
if not runs:
    print('目前沒有 run_* 結果。')
else:
    latest = runs[-1]
    print('最新單次結果：', latest)
    summary_path = latest / 'final_metrics_summary.csv'
    if summary_path.exists():
        print('摘要檔：', summary_path)
        display(pd.read_csv(summary_path))
    metadata_path = latest / 'run_metadata.json'
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        print(json.dumps(metadata, ensure_ascii=False, indent=2)[:2000])
    plot_paths = [
        latest / 'plots' / 'optimizer_lambda_tradeoff.png',
        latest / 'plots' / 'top_configs_by_auc.png',
        latest / 'plots' / 'feature_count_vs_auc.png',
    ]
    for plot_path in plot_paths:
        if plot_path.exists():
            print('可視化圖檔：', plot_path)
            display(Image(filename=str(plot_path)))


## 五、跑正式 Kaggle 資料

預設 `RUN_REAL = False`。確認要跑時再改成 `True`。


In [4]:
from pathlib import Path
import subprocess
import sys

RUN_REAL = True
REAL_PRESET = 'standard'  # smoke / quick / standard
SEARCH_FEATURES = 150  # Word 題目寫 150；如果想用 preset 預設值，改成 None
PROXY_ROWS = 20000
DEVICE = 'cpu'  # 有 CUDA 且 xgboost 支援時才改成 'cuda'
DATA_PATH = Path('data/raw/train_transaction.csv')

if not RUN_REAL:
    print('正式資料執行預設關閉。要跑時請把 RUN_REAL 改成 True。')
elif not DATA_PATH.exists():
    raise FileNotFoundError('找不到 train_transaction.csv。請先放到 data/raw/。' + ' ' + str(DATA_PATH))
else:
    print('資料路徑：', DATA_PATH)
    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        '--mode', 'real',
        '--data-path', str(DATA_PATH),
        '--preset', REAL_PRESET,
        '--device', DEVICE,
    ]
    if SEARCH_FEATURES is not None:
        cmd += ['--search-features', str(SEARCH_FEATURES)]
    if PROXY_ROWS is not None:
        cmd += ['--proxy-rows', str(PROXY_ROWS)]
    print('執行指令：', ' '.join(cmd))
    run_stream(cmd)


資料路徑： data\raw\train_transaction.csv
執行指令： d:\Python310\python.exe -u run_experiment.py --mode real --data-path data\raw\train_transaction.csv --preset standard --device cpu --search-features 150 --proxy-rows 20000
[1/5] Loading data: mode=real, preset=standard
[2/5] Checking preprocessing cache
      cache hit: D:\ML期末\GA_SA_feature_selection_project\data\cache\2e699db744c109c3.npz (0.0s)
      rows=20000, original_features=392, search_features=150
[3/5] Lambda 1/4: 0.0
      RandomSearch 1/300: best_auc=0.8450, features=84
      RandomSearch 30/300: best_auc=0.8510, features=88
      RandomSearch 60/300: best_auc=0.8510, features=88
      RandomSearch 90/300: best_auc=0.8510, features=88
      RandomSearch 120/300: best_auc=0.8542, features=91
      RandomSearch 150/300: best_auc=0.8621, features=85
      RandomSearch 180/300: best_auc=0.8621, features=85
      RandomSearch 210/300: best_auc=0.8621, features=85
      RandomSearch 240/300: best_auc=0.8621, features=85
      RandomSear

## 六、跑多 seed 批次實驗

預設 `RUN_MULTI_SEED = False`。確認要跑時再改成 `True`。


In [5]:
import subprocess
import sys

RUN_MULTI_SEED = True
SEEDS = '7,42,2026'
MULTI_PRESET = 'standard'  # 時間夠就用 standard；只想快速補穩定性才改 quick
MODE = 'real'  # 正式資料用 real；只測流程才改 synthetic
SEARCH_FEATURES = 150  # Word 題目寫 150
PROXY_ROWS = 20000
DEVICE = 'cuda'  # 有 CUDA 且 xgboost 支援時才改成 'cuda'
DATA_PATH = 'data/raw/train_transaction.csv'

if not RUN_MULTI_SEED:
    print('多 seed 批次執行預設關閉。要跑時請把 RUN_MULTI_SEED 改成 True。')
else:
    cmd = [
        sys.executable,
        '-u',
        'run_multi_seed.py',
        '--seeds', SEEDS,
        '--preset', MULTI_PRESET,
        '--mode', MODE,
        '--device', DEVICE,
    ]
    if MODE == 'real':
        cmd += ['--data-path', DATA_PATH]
    if SEARCH_FEATURES is not None:
        cmd += ['--search-features', str(SEARCH_FEATURES)]
    if PROXY_ROWS is not None:
        cmd += ['--proxy-rows', str(PROXY_ROWS)]
    print('執行指令：', ' '.join(cmd))
    run_stream(cmd)


執行指令： d:\Python310\python.exe -u run_multi_seed.py --seeds 7,42,2026 --preset standard --mode real --device cuda --data-path data/raw/train_transaction.csv --search-features 150 --proxy-rows 20000
=== Seed 1/3: 7 ===
[1/5] Loading data: mode=real, preset=standard
[2/5] Checking preprocessing cache
      cache miss/rebuilt: data\cache\1f936cc009d3b226.npz (60.1s)
      rows=20000, original_features=392, search_features=150
[3/5] Lambda 1/4: 0.0
d:\Python310\lib\site-packages\xgboost\core.py:751: UserWarning: [23:18:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return fu

## 七、查看最新彙總結果


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

aggregates = sorted(Path('results').glob('aggregate_*'))
if not aggregates:
    print('目前沒有 aggregate_* 彙總結果。')
else:
    latest = aggregates[-1]
    print('最新彙總結果：', latest)
    runs_path = latest / 'included_runs.csv'
    summary_path = latest / 'aggregate_summary.csv'
    metadata_path = latest / 'aggregate_metadata.json'
    if runs_path.exists():
        print('包含的 runs：')
        display(pd.read_csv(runs_path))
    if summary_path.exists():
        print('彙總摘要檔：')
        display(pd.read_csv(summary_path))
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        print(json.dumps(metadata, ensure_ascii=False, indent=2)[:2000])
